# Module 4. Milestone 1

## Step 1: Data loading and filtering

In this step, we prepare the dataset and keep only orders with at least 5 purchased items.
First we load the dataset, then we keep only these type of orders and we build the filtered dataset

In [1]:
import pandas as pd

DATA_PATH = "../data/module_2/box_builder_dataset/feature_frame.csv"
TARGET = "outcome"

df = pd.read_csv(DATA_PATH, parse_dates=["order_date", "created_at"])

items_bought_per_order = (
    df[df[TARGET] == 1.0]
    .groupby("order_id")["variant_id"]
    .count()
)
valid_orders = items_bought_per_order[items_bought_per_order >= 5].index
df_filtered = df[df["order_id"].isin(valid_orders)].reset_index(drop=True)

orders_before = df["order_id"].nunique()
orders_after = df_filtered["order_id"].nunique()
rows_before = len(df)
rows_after = len(df_filtered)

print(f"Rows: {rows_before:,} -> {rows_after:,}")
print(f"Orders: {orders_before:,} -> {orders_after:,}")
print(f"Removed orders: {orders_before - orders_after:,}")
print(f"Share of orders kept: {orders_after / orders_before:.2%}")

df_filtered.head(3)

Rows: 2,880,549 -> 2,163,953
Orders: 3,446 -> 2,603
Removed orders: 843
Share of orders kept: 75.54%


,variant_id,product_type,order_id,user_id,created_at,order_date,user_order_seq,outcome,ordered_before,abandoned_before,...,count_children,count_babies,count_pets,people_ex_baby,days_since_purchase_variant_id,avg_days_to_buy_variant_id,std_days_to_buy_variant_id,days_since_purchase_product_type,avg_days_to_buy_product_type,std_days_to_buy_product_type
0,33826472919172,ricepastapulses,2807985930372,3482464092292,2020-10-05 16:46:19,2020-10-05,3,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
1,33826472919172,ricepastapulses,2808027644036,3466586718340,2020-10-05 17:59:51,2020-10-05,2,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
2,33826472919172,ricepastapulses,2808099078276,3481384026244,2020-10-05 20:08:53,2020-10-05,4,0.0,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618


## Step 2: Feature engineering
- Numeric features stay as numbers
- `product_type` to one-hot encoded.
- The target is `outcome` (bought/ not bought)

In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

TARGET = "outcome"

numeric_features = [
    "ordered_before",
    "abandoned_before",
    "active_snoozed",
    "set_as_regular",
    "normalised_price",
    "discount_pct",
    "global_popularity",
    "count_adults",
    "count_children",
    "count_babies",
    "count_pets",
    "people_ex_baby",
    "days_since_purchase_variant_id",
    "avg_days_to_buy_variant_id",
    "std_days_to_buy_variant_id",
    "days_since_purchase_product_type",
    "avg_days_to_buy_product_type",
    "std_days_to_buy_product_type",
    "user_order_seq",
]
categorical_features = ["product_type"]

feature_columns = numeric_features + categorical_features
X = df_filtered[feature_columns].copy()
y = df_filtered[TARGET].astype(int)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

X_transformed = preprocessor.fit_transform(X)

print(f"Filtered rows: {len(df_filtered):,}")
print(f"X shape (raw): {X.shape}")
print(f"y shape: {y.shape}")
print(f"X shape (transformed): {X_transformed.shape}")
print(f"Positive class rate: {y.mean():.2%}")

Filtered rows: 2,163,953
X shape (raw): (2163953, 20)
y shape: (2163953,)
X shape (transformed): (2163953, 81)
Positive class rate: 1.45%


We get X to train, and a target Y to predict. As we see, our data has very few positive classes (1.45%) 

## Step 3: Train / validation / test (time)

The same as in Module 3, if we split randomly, the model could learn from future insights or behaviors on things that, in this case, doesn't have any sense. 
So, we split by time:
- **Train (60%)**: older orders.
- **Validation (20%)**: later orders.
- **Test (20%)**: most recent orders.

In [3]:
orders_by_date = (
    df_filtered[["order_id", "order_date"]]
    .drop_duplicates()
    .sort_values("order_date")
    .reset_index(drop=True)
)

n_orders = len(orders_by_date)
train_cut_idx = int(n_orders * 0.60)
val_cut_idx = int(n_orders * 0.80)

train_cutoff_date = orders_by_date.iloc[train_cut_idx]["order_date"]
val_cutoff_date = orders_by_date.iloc[val_cut_idx]["order_date"]

train_mask = df_filtered["order_date"] < train_cutoff_date
val_mask = (df_filtered["order_date"] >= train_cutoff_date) & (
    df_filtered["order_date"] < val_cutoff_date
)
test_mask = df_filtered["order_date"] >= val_cutoff_date

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

print("Date boundaries:")
print(f"- Train until: {train_cutoff_date.date()}")
print(f"- Validation until: {val_cutoff_date.date()}")

print("\nRows per split:")
print(f"- Train: {len(X_train):,} ({len(X_train) / len(X):.2%})")
print(f"- Validation: {len(X_val):,} ({len(X_val) / len(X):.2%})")
print(f"- Test: {len(X_test):,} ({len(X_test) / len(X):.2%})")

print("\nPositive rate per split:")
print(f"- Train: {y_train.mean():.2%}")
print(f"- Validation: {y_val.mean():.2%}")
print(f"- Test: {y_test.mean():.2%}")

Date boundaries:
- Train until: 2021-01-26
- Validation until: 2021-02-15

Rows per split:
- Train: 1,207,702 (55.81%)
- Validation: 453,596 (20.96%)
- Test: 502,655 (23.23%)

Positive rate per split:
- Train: 1.53%
- Validation: 1.36%
- Test: 1.32%


We have similar positive rates in all splits (this is a good sing, right?)

## Step 4: Baseline model (Logistic Regression)

Now we train a simple baseline model as a reference point before more difficult non-linear models. This is because, we would only prefer a more difficult model if it clearly improves our result, if not worth it, we should maintain the easy model.
Also, doing this, we could see if the signal of the data is mostly linear or we need non-linear models.

We reuse the best setup from Module 3 (`L2` regularization with `C=0.1`) and evaluate PR-AUC, which is maybe useful here due to class imbalance. Also, we prove ROC-AUC.

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

lr_preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

lr_baseline = Pipeline(
    steps=[
        ("preprocessor", lr_preprocessor),
        (
            "model",
            LogisticRegression(
                penalty="l2",
                C=0.1,
                solver="lbfgs",
                class_weight="balanced",
                max_iter=3000,
                random_state=42,
            ),
        ),
    ]
)

lr_baseline.fit(X_train, y_train)

val_proba = lr_baseline.predict_proba(X_val)[:, 1]
test_proba = lr_baseline.predict_proba(X_test)[:, 1]

val_pr_auc = average_precision_score(y_val, val_proba)
test_pr_auc = average_precision_score(y_test, test_proba)
val_roc_auc = roc_auc_score(y_val, val_proba)
test_roc_auc = roc_auc_score(y_test, test_proba)

print("Baseline Logistic Regression results:")
print(f"- Validation PR-AUC: {val_pr_auc:.4f}")
print(f"- Test PR-AUC: {test_pr_auc:.4f}")
print(f"- Validation ROC-AUC: {val_roc_auc:.4f}")
print(f"- Test ROC-AUC: {test_roc_auc:.4f}")

baseline_results = {
    "model": "logistic_regression_l2_c0.1_balanced",
    "val_pr_auc": val_pr_auc,
    "test_pr_auc": test_pr_auc,
    "val_roc_auc": val_roc_auc,
    "test_roc_auc": test_roc_auc,
}

baseline_results

Baseline Logistic Regression results:
- Validation PR-AUC: 0.1431
- Test PR-AUC: 0.1527
- Validation ROC-AUC: 0.8352
- Test ROC-AUC: 0.8299


{'model': 'logistic_regression_l2_c0.1_balanced',
 'val_pr_auc': 0.14306706126582033,
 'test_pr_auc': 0.15273836411805486,
 'val_roc_auc': 0.8352471113503817,
 'test_roc_auc': 0.8299161639184548}

PR-AUC is moderate, so we can still improve maybe comparing with non-linear models.
The ROC-AUC of ~0.83 says the model can rank in a good level the positives, but maybe in our model is not ennough by itself because our positives are low (1.45%).

## Step 5: Decision Tree

We test a Decision Tree and change `max_depth` using different depths (3,5,7,10,14) using time-based cross-validation on training phase.
Then we retrain with the best depth on full train and evaluate on validation and test and compare against the logistic regression basline.

In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier


def build_tree(depth: int) -> Pipeline:
    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                DecisionTreeClassifier(
                    max_depth=depth,
                    min_samples_leaf=50,
                    class_weight="balanced",
                    random_state=42,
                ),
            ),
        ]
    )


depth_grid = [3, 5, 7, 10, 14]
train_orders = (
    df_filtered.loc[train_mask, ["order_id", "order_date"]]
    .drop_duplicates()
    .sort_values("order_date")
    .reset_index(drop=True)
)

train_order_ids = df_filtered.loc[train_mask, "order_id"].reset_index(drop=True)
X_train_r = X_train.reset_index(drop=True)
y_train_r = y_train.reset_index(drop=True)

tscv = TimeSeriesSplit(n_splits=3)
cv_rows = []

for depth in depth_grid:
    fold_scores = []
    model = build_tree(depth)

    for fit_idx, cv_idx in tscv.split(train_orders):
        fit_orders = set(train_orders.iloc[fit_idx]["order_id"])
        cv_orders = set(train_orders.iloc[cv_idx]["order_id"])

        fit_mask = train_order_ids.isin(fit_orders).values
        cv_mask = train_order_ids.isin(cv_orders).values

        model.fit(X_train_r.loc[fit_mask], y_train_r.loc[fit_mask])
        cv_proba = model.predict_proba(X_train_r.loc[cv_mask])[:, 1]
        fold_scores.append(average_precision_score(y_train_r.loc[cv_mask], cv_proba))

    cv_rows.append(
        {
            "max_depth": depth,
            "cv_pr_auc_mean": float(np.mean(fold_scores)),
            "cv_pr_auc_std": float(np.std(fold_scores)),
        }
    )

cv_results_tree = pd.DataFrame(cv_rows).sort_values(
    "cv_pr_auc_mean", ascending=False
).reset_index(drop=True)
best_depth = int(cv_results_tree.loc[0, "max_depth"])

best_tree = build_tree(best_depth)
best_tree.fit(X_train, y_train)

tree_val_proba = best_tree.predict_proba(X_val)[:, 1]
tree_test_proba = best_tree.predict_proba(X_test)[:, 1]

tree_val_pr_auc = average_precision_score(y_val, tree_val_proba)
tree_test_pr_auc = average_precision_score(y_test, tree_test_proba)
tree_val_roc_auc = roc_auc_score(y_val, tree_val_proba)
tree_test_roc_auc = roc_auc_score(y_test, tree_test_proba)

print("Decision Tree CV results (train only):")
print(cv_results_tree.to_string(index=False))
print(f"\nSelected max_depth: {best_depth}")

print("\nDecision Tree holdout performance:")
print(f"- Validation PR-AUC: {tree_val_pr_auc:.4f}")
print(f"- Test PR-AUC: {tree_test_pr_auc:.4f}")
print(f"- Validation ROC-AUC: {tree_val_roc_auc:.4f}")
print(f"- Test ROC-AUC: {tree_test_roc_auc:.4f}")

comparison_step5 = pd.DataFrame(
    [
        {
            "model": "logistic_baseline",
            "val_pr_auc": baseline_results["val_pr_auc"],
            "test_pr_auc": baseline_results["test_pr_auc"],
            "val_roc_auc": baseline_results["val_roc_auc"],
            "test_roc_auc": baseline_results["test_roc_auc"],
        },
        {
            "model": f"decision_tree_depth_{best_depth}",
            "val_pr_auc": tree_val_pr_auc,
            "test_pr_auc": tree_test_pr_auc,
            "val_roc_auc": tree_val_roc_auc,
            "test_roc_auc": tree_test_roc_auc,
        },
    ]
)

comparison_step5

Decision Tree CV results (train only):
 max_depth  cv_pr_auc_mean  cv_pr_auc_std
        10        0.156955       0.004014
         7        0.154259       0.005390
        14        0.152359       0.002941
         5        0.141853       0.007584
         3        0.082621       0.007307

Selected max_depth: 10

Decision Tree holdout performance:
- Validation PR-AUC: 0.1537
- Test PR-AUC: 0.1670
- Validation ROC-AUC: 0.8202
- Test ROC-AUC: 0.8202


,model,val_pr_auc,test_pr_auc,val_roc_auc,test_roc_auc
0,logistic_baseline,0.143067,0.152738,0.835247,0.829916
1,decision_tree_depth_10,0.153660,0.166978,0.820216,0.820155


We obtain that the best depth is 10 because it gave the highest average PR-AUC. The Decision Tree gives an improvement in PR-AUC over Logistic Regression and ROC-AUC is lower than Logistic, but for this imbalance in data, maybe PR-AUC is more important.

## Step 6: Random Forest

We test a Random Forest with more trees. We try estimators of 80 or 160 trees and depth of 8 or 12, and time-based CV on train. At the end we pick the best combination, train it, and evaluate it on validation/test to compare with both previous models.

With Random Forest we could obtain more robust, less noisy and better generalization of the predictions.

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_param_grid = [
    {"n_estimators": 80, "max_depth": 8},
    {"n_estimators": 80, "max_depth": 12},
    {"n_estimators": 160, "max_depth": 8},
    {"n_estimators": 160, "max_depth": 12},
]

rf_orders = train_orders.tail(min(900, len(train_orders))).reset_index(drop=True)
rf_order_ids = set(rf_orders["order_id"])
rf_subset_mask = train_order_ids.isin(rf_order_ids).values

X_rf = X_train_r.loc[rf_subset_mask].reset_index(drop=True)
y_rf = y_train_r.loc[rf_subset_mask].reset_index(drop=True)
rf_train_order_ids = train_order_ids.loc[rf_subset_mask].reset_index(drop=True)

tscv_rf = TimeSeriesSplit(n_splits=3)
rf_cv_rows = []

for params in rf_param_grid:
    fold_scores = []
    rf_model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=params["n_estimators"],
                    max_depth=params["max_depth"],
                    min_samples_leaf=20,
                    class_weight="balanced_subsample",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    for fit_idx, cv_idx in tscv_rf.split(rf_orders):
        fit_orders = set(rf_orders.iloc[fit_idx]["order_id"])
        cv_orders = set(rf_orders.iloc[cv_idx]["order_id"])

        fit_mask = rf_train_order_ids.isin(fit_orders).values
        cv_mask = rf_train_order_ids.isin(cv_orders).values

        rf_model.fit(X_rf.loc[fit_mask], y_rf.loc[fit_mask])
        cv_proba = rf_model.predict_proba(X_rf.loc[cv_mask])[:, 1]
        fold_scores.append(average_precision_score(y_rf.loc[cv_mask], cv_proba))

    rf_cv_rows.append(
        {
            "n_estimators": params["n_estimators"],
            "max_depth": params["max_depth"],
            "cv_pr_auc_mean": float(np.mean(fold_scores)),
            "cv_pr_auc_std": float(np.std(fold_scores)),
        }
    )

rf_cv_results = pd.DataFrame(rf_cv_rows).sort_values(
    "cv_pr_auc_mean", ascending=False
).reset_index(drop=True)

best_rf_params = {
    "n_estimators": int(rf_cv_results.loc[0, "n_estimators"]),
    "max_depth": int(rf_cv_results.loc[0, "max_depth"]),
}

best_rf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=best_rf_params["n_estimators"],
                max_depth=best_rf_params["max_depth"],
                min_samples_leaf=20,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

best_rf.fit(X_rf, y_rf)

rf_val_proba = best_rf.predict_proba(X_val)[:, 1]
rf_test_proba = best_rf.predict_proba(X_test)[:, 1]

rf_val_pr_auc = average_precision_score(y_val, rf_val_proba)
rf_test_pr_auc = average_precision_score(y_test, rf_test_proba)
rf_val_roc_auc = roc_auc_score(y_val, rf_val_proba)
rf_test_roc_auc = roc_auc_score(y_test, rf_test_proba)

print("Random Forest CV results (train subset):")
print(rf_cv_results.to_string(index=False))
print(f"\nSelected params: {best_rf_params}")

print("\nRandom Forest holdout performance:")
print(f"- Validation PR-AUC: {rf_val_pr_auc:.4f}")
print(f"- Test PR-AUC: {rf_test_pr_auc:.4f}")
print(f"- Validation ROC-AUC: {rf_val_roc_auc:.4f}")
print(f"- Test ROC-AUC: {rf_test_roc_auc:.4f}")

comparison_step6 = pd.DataFrame(
    [
        {
            "model": "logistic_baseline",
            "val_pr_auc": baseline_results["val_pr_auc"],
            "test_pr_auc": baseline_results["test_pr_auc"],
        },
        {
            "model": f"decision_tree_depth_{best_depth}",
            "val_pr_auc": tree_val_pr_auc,
            "test_pr_auc": tree_test_pr_auc,
        },
        {
            "model": (
                "random_forest_"
                f"n{best_rf_params['n_estimators']}_"
                f"d{best_rf_params['max_depth']}"
            ),
            "val_pr_auc": rf_val_pr_auc,
            "test_pr_auc": rf_test_pr_auc,
        },
    ]
)

comparison_step6

Random Forest CV results (train subset):
 n_estimators  max_depth  cv_pr_auc_mean  cv_pr_auc_std
          160         12        0.130698       0.003674
           80         12        0.129515       0.002133
          160          8        0.125571       0.003698
           80          8        0.120854       0.004118

Selected params: {'n_estimators': 160, 'max_depth': 12}

Random Forest holdout performance:
- Validation PR-AUC: 0.1552
- Test PR-AUC: 0.1686
- Validation ROC-AUC: 0.8361
- Test ROC-AUC: 0.8344


,model,val_pr_auc,test_pr_auc
0,logistic_baseline,0.143067,0.152738
1,decision_tree_depth_10,0.153660,0.166978
2,random_forest_n160_d12,0.155196,0.168613


The best config. we obtained is 160 estimators and depth of 12. Compared with the previous models, we have a gain in PR-AUC compared to both of them, and also stronger in ROC_AUC than Single tree.
This tells us that ensemble learning is helping and it is worth testing Boosting trees next.

## Step 7: Gradient Boosting + final selection

We test Gradient Boosting with parameters `max_iter`, `learning_rate`, and `max_depth`, using time-based CV on a recent train subset.
Then we retrain the best configuration, compare all candidate models, and select the final winner for Milestone 2.

In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder


gb_param_grid = [
    {"max_iter": 120, "learning_rate": 0.05, "max_depth": 4},
    {"max_iter": 120, "learning_rate": 0.10, "max_depth": 4},
    {"max_iter": 180, "learning_rate": 0.10, "max_depth": 6},
]

gb_preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        (
            "cat",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
            categorical_features,
        ),
    ]
)

gb_orders = train_orders.tail(min(700, len(train_orders))).reset_index(drop=True)
gb_order_ids = set(gb_orders["order_id"])
gb_subset_mask = train_order_ids.isin(gb_order_ids).values

X_gb = X_train_r.loc[gb_subset_mask].reset_index(drop=True)
y_gb = y_train_r.loc[gb_subset_mask].reset_index(drop=True)
gb_train_order_ids = train_order_ids.loc[gb_subset_mask].reset_index(drop=True)

tscv_gb = TimeSeriesSplit(n_splits=3)
gb_cv_rows = []

for params in gb_param_grid:
    fold_scores = []
    gb_model = Pipeline(
        steps=[
            ("preprocessor", gb_preprocessor),
            (
                "model",
                HistGradientBoostingClassifier(
                    max_iter=params["max_iter"],
                    learning_rate=params["learning_rate"],
                    max_depth=params["max_depth"],
                    random_state=42,
                ),
            ),
        ]
    )

    for fit_idx, cv_idx in tscv_gb.split(gb_orders):
        fit_orders = set(gb_orders.iloc[fit_idx]["order_id"])
        cv_orders = set(gb_orders.iloc[cv_idx]["order_id"])

        fit_mask = gb_train_order_ids.isin(fit_orders).values
        cv_mask = gb_train_order_ids.isin(cv_orders).values

        gb_model.fit(X_gb.loc[fit_mask], y_gb.loc[fit_mask])
        cv_proba = gb_model.predict_proba(X_gb.loc[cv_mask])[:, 1]
        fold_scores.append(average_precision_score(y_gb.loc[cv_mask], cv_proba))

    gb_cv_rows.append(
        {
            "max_iter": params["max_iter"],
            "learning_rate": params["learning_rate"],
            "max_depth": params["max_depth"],
            "cv_pr_auc_mean": float(np.mean(fold_scores)),
            "cv_pr_auc_std": float(np.std(fold_scores)),
        }
    )

gb_cv_results = pd.DataFrame(gb_cv_rows).sort_values(
    "cv_pr_auc_mean", ascending=False
).reset_index(drop=True)

best_gb_params = {
    "max_iter": int(gb_cv_results.loc[0, "max_iter"]),
    "learning_rate": float(gb_cv_results.loc[0, "learning_rate"]),
    "max_depth": int(gb_cv_results.loc[0, "max_depth"]),
}

best_gb = Pipeline(
    steps=[
        ("preprocessor", gb_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                max_iter=best_gb_params["max_iter"],
                learning_rate=best_gb_params["learning_rate"],
                max_depth=best_gb_params["max_depth"],
                random_state=42,
            ),
        ),
    ]
)

best_gb.fit(X_gb, y_gb)

gb_val_proba = best_gb.predict_proba(X_val)[:, 1]
gb_test_proba = best_gb.predict_proba(X_test)[:, 1]

gb_val_pr_auc = average_precision_score(y_val, gb_val_proba)
gb_test_pr_auc = average_precision_score(y_test, gb_test_proba)
gb_val_roc_auc = roc_auc_score(y_val, gb_val_proba)
gb_test_roc_auc = roc_auc_score(y_test, gb_test_proba)

print("Gradient Boosting CV results (train subset):")
print(gb_cv_results.to_string(index=False))
print(f"\nSelected params: {best_gb_params}")

print("\nGradient Boosting holdout performance:")
print(f"- Validation PR-AUC: {gb_val_pr_auc:.4f}")
print(f"- Test PR-AUC: {gb_test_pr_auc:.4f}")
print(f"- Validation ROC-AUC: {gb_val_roc_auc:.4f}")
print(f"- Test ROC-AUC: {gb_test_roc_auc:.4f}")

model_summary = pd.DataFrame(
    [
        {
            "model": "logistic_regression",
            "val_pr_auc": baseline_results["val_pr_auc"],
            "test_pr_auc": baseline_results["test_pr_auc"],
            "val_roc_auc": baseline_results["val_roc_auc"],
            "test_roc_auc": baseline_results["test_roc_auc"],
        },
        {
            "model": f"decision_tree_depth_{best_depth}",
            "val_pr_auc": tree_val_pr_auc,
            "test_pr_auc": tree_test_pr_auc,
            "val_roc_auc": tree_val_roc_auc,
            "test_roc_auc": tree_test_roc_auc,
        },
        {
            "model": (
                "random_forest_"
                f"n{best_rf_params['n_estimators']}_"
                f"d{best_rf_params['max_depth']}"
            ),
            "val_pr_auc": rf_val_pr_auc,
            "test_pr_auc": rf_test_pr_auc,
            "val_roc_auc": rf_val_roc_auc,
            "test_roc_auc": rf_test_roc_auc,
        },
        {
            "model": (
                "hist_gradient_boosting_"
                f"i{best_gb_params['max_iter']}_"
                f"lr{best_gb_params['learning_rate']}_"
                f"d{best_gb_params['max_depth']}"
            ),
            "val_pr_auc": gb_val_pr_auc,
            "test_pr_auc": gb_test_pr_auc,
            "val_roc_auc": gb_val_roc_auc,
            "test_roc_auc": gb_test_roc_auc,
        },
    ]
).sort_values("val_pr_auc", ascending=False).reset_index(drop=True)

winner = model_summary.iloc[0]
runner_up = model_summary.iloc[1]

print("\nModel ranking by validation PR-AUC:")
print(model_summary.to_string(index=False))
print("\nSelected model:")
print(f"- {winner['model']}")
print(f"- Validation PR-AUC: {winner['val_pr_auc']:.4f}")
print(f"- Test PR-AUC: {winner['test_pr_auc']:.4f}")
print(f"- Margin vs 2nd model on validation PR-AUC: {winner['val_pr_auc'] - runner_up['val_pr_auc']:.4f}")

final_model_name = winner["model"]
final_val_pr_auc = float(winner["val_pr_auc"])
final_test_pr_auc = float(winner["test_pr_auc"])

comparison_step7 = model_summary[["model", "val_pr_auc", "test_pr_auc"]].copy()
comparison_step7

Gradient Boosting CV results (train subset):
 max_iter  learning_rate  max_depth  cv_pr_auc_mean  cv_pr_auc_std
      120           0.05          4        0.173835       0.016946
      120           0.10          4        0.166925       0.014784
      180           0.10          6        0.160332       0.019726

Selected params: {'max_iter': 120, 'learning_rate': 0.05, 'max_depth': 4}

Gradient Boosting holdout performance:
- Validation PR-AUC: 0.1719
- Test PR-AUC: 0.1839
- Validation ROC-AUC: 0.8439
- Test ROC-AUC: 0.8435

Model ranking by validation PR-AUC:
                                model  val_pr_auc  test_pr_auc  val_roc_auc  test_roc_auc
hist_gradient_boosting_i120_lr0.05_d4    0.171864     0.183947     0.843913      0.843532
               random_forest_n160_d12    0.155196     0.168613     0.836148      0.834365
               decision_tree_depth_10    0.153660     0.166978     0.820216      0.820155
                  logistic_regression    0.143067     0.152738     0.8352

,model,val_pr_auc,test_pr_auc
0,hist_gradient_boosting_i120_lr0.05_d4,0.171864,0.183947
1,random_forest_n160_d12,0.155196,0.168613
2,decision_tree_depth_10,0.153660,0.166978
3,logistic_regression,0.143067,0.152738


We obtain that the best was to do 120 iterations, a learning rate of 0.05 and a max depth of 4.
We see this is the strongest model so far: a clear PR-AUC gain over Random Forest and also best ROC-AUC among all models.
Improvement takes place in both validation and test, so this tells it is not just overfitting noise.
So, it is worth to say that HistGradientBoosting is the leading candidate.

## Step 8: Probability calibration

We will try to improve probability quality of our final model:

We take our winning model Gradient Boosting, (already trained), we generate a non-calibrated version and 2 calibrated versions (sigmoid, isotonic), and compare 3 versions in test with PR-AUC and Brier score (the lower the better).

This helps us not only to get good ranking (PR-AUC), but also to make probabilities more confident for further decisions. We will keep the calibration only if it improves Brier score and doesn't drop PR-AUC substantially.

In [11]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import brier_score_loss

selected_model = Pipeline(
    steps=[
        ("preprocessor", gb_preprocessor),
        (
            "model",
            HistGradientBoostingClassifier(
                max_iter=best_gb_params["max_iter"],
                learning_rate=best_gb_params["learning_rate"],
                max_depth=best_gb_params["max_depth"],
                random_state=42,
            ),
        ),
    ]
)

selected_model.fit(X_train, y_train)

uncal_test_proba = selected_model.predict_proba(X_test)[:, 1]

cal_sigmoid = CalibratedClassifierCV(estimator=selected_model, method="sigmoid", cv="prefit")
cal_sigmoid.fit(X_val, y_val)
sigmoid_test_proba = cal_sigmoid.predict_proba(X_test)[:, 1]

cal_isotonic = CalibratedClassifierCV(estimator=selected_model, method="isotonic", cv="prefit")
cal_isotonic.fit(X_val, y_val)
isotonic_test_proba = cal_isotonic.predict_proba(X_test)[:, 1]

calibration_results = pd.DataFrame(
    [
        {
            "variant": "uncalibrated",
            "test_pr_auc": average_precision_score(y_test, uncal_test_proba),
            "test_brier": brier_score_loss(y_test, uncal_test_proba),
        },
        {
            "variant": "sigmoid",
            "test_pr_auc": average_precision_score(y_test, sigmoid_test_proba),
            "test_brier": brier_score_loss(y_test, sigmoid_test_proba),
        },
        {
            "variant": "isotonic",
            "test_pr_auc": average_precision_score(y_test, isotonic_test_proba),
            "test_brier": brier_score_loss(y_test, isotonic_test_proba),
        },
    ]
).sort_values("test_brier").reset_index(drop=True)

best_calibrated = calibration_results.iloc[0]
uncalibrated_row = calibration_results[calibration_results["variant"] == "uncalibrated"].iloc[0]

pr_auc_drop = uncalibrated_row["test_pr_auc"] - best_calibrated["test_pr_auc"]
brier_gain = uncalibrated_row["test_brier"] - best_calibrated["test_brier"]

use_calibration = (brier_gain > 0.0) and (pr_auc_drop <= 0.003)

print("Calibration comparison on test:")
print(calibration_results.to_string(index=False))
print("\nDecision:")
print(f"- Best calibrated variant by Brier: {best_calibrated['variant']}")
print(f"- Brier gain vs uncalibrated: {brier_gain:.6f}")
print(f"- PR-AUC drop vs uncalibrated: {pr_auc_drop:.6f}")
print(f"- Use calibration: {use_calibration}")

final_probability_model = best_calibrated["variant"] if use_calibration else "uncalibrated"
final_probability_model

Calibration comparison on test:
     variant  test_pr_auc  test_brier
     sigmoid     0.188132    0.011726
    isotonic     0.177946    0.011729
uncalibrated     0.188132    0.011761

Decision:
- Best calibrated variant by Brier: sigmoid
- Brier gain vs uncalibrated: 0.000035
- PR-AUC drop vs uncalibrated: 0.000000
- Use calibration: True


'sigmoid'

The results tell it is safe to use calibration, with sigmoid variant. The gain in prob. quality would be really small, but the PR-AUC remains the same, so ranking quality doesn't get worse but gain in prob.quality would be slightly better.

## Step 9: Conclusions and handoff decision

### What worked
- Non-linear models improved PR-AUC over Logistic Regression.
- Tree ensembles (many trees) performed better than a single Tree.
- HistGradientBoosting gave us the best validation and test PR-AUC.

### What did not work as well
- Logistic Regression remained a useful baseline but not performed as well as non-linear models.
- Decision Tree improved over baseline, but less than ensemble methods.

### Final Milestone 1 decision
- We move to Milestone 2 with the selected HistGradientBoosting configuration from Step 7.
- We use the calibrated probabilities 